# 🏦 Banking Churn Case Study
### Jack Henry Associates — Data Science Talk

---

This notebook walks through a complete data science workflow applied to **banking customer churn**.
All heavy lifting lives in `churn_utils.py` — cells here focus on **results and narrative**.

**Workflow:**
```
Data  →  Label  →  Explore  →  Partition (by date)  →  Feature Engineering
  →  Train (AutoGluon)  →  Evaluate (ML + Business)  →  SHAP  →  DALEX  →  Local Explain
```

**Key JH-relevant features:** DD velocity, digital logins, balance patterns, product depth, service friction

> **Dependencies:** `pip install autogluon.tabular shap dalex catboost matplotlib seaborn pandas numpy`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import churn_utils as cu
print('✅ churn_utils loaded')

---
## 🔍 Opening Hook: The Mystery Customer

Before any data — here's a real customer profile. **Will they close their account in 90 days?**

We'll answer this with a full SHAP + DALEX explanation at the end.

In [ ]:
df_raw = cu.generate_banking_data(n_customers=6000)
mystery = cu.get_mystery_customer(df_raw)
cu.print_mystery_customer(mystery)

---
## 1. The Data

Banking customer data — one row per customer, snapshot at observation date.

| Feature Group | Key Fields | Signal |
|---|---|---|
| **DD Velocity** | dd_count_30d, dd_amount_30d, days_since_last_dd | Primary bank? |
| **Digital Engagement** | digital_login_30d, mobile_login_30d, days_since_last_login | Active user? |
| **Balance Patterns** | avg_daily_balance, balance_volatility, overdraft_count_12m | Financial health |
| **Products** | product_count, tenure_months, has_savings, has_loan | Relationship depth |
| **Service Friction** | nsf_count_12m, fee_waiver_req_12m, cust_service_calls_6m | Dissatisfaction |


In [ ]:
df_raw.head(5)

In [ ]:
print(f'Shape     : {df_raw.shape}')
print(f'Date range: {df_raw.signup_date.min().date()} → {df_raw.signup_date.max().date()}')
print(f'Churn rate: {df_raw.churn_90d.mean()*100:.1f}%  ({df_raw.churn_90d.sum():,} churners)')

---
## 2. Labeling

Getting the label right is half the battle. **Poor labeling = poor model**, regardless of algorithm sophistication.

**Definition:** Account closure OR complete dormancy (zero transactions + zero logins) within 90 days of snapshot.

In [ ]:
cu.explain_labeling()

---
## 3. Exploratory Data Analysis

In [ ]:
cu.plot_eda(df_raw)

In [ ]:
cu.plot_correlation_heatmap(df_raw)

**Key findings:** No primary DD → 3–4× higher churn. Under 5 logins/month → strong leading indicator. Each additional product reduces churn meaningfully.

---
## 4. Temporal Partitioning

> ⚠️ **Most common mistake:** random train/test splits on time-series data cause **data leakage**.

Always split by date — the model must only see customers who signed up *before* the cutoff.

In [ ]:
cu.plot_temporal_split(df_raw)

In [ ]:
train_raw, val_raw, test_raw = cu.temporal_split(df_raw)

---
## 5. Feature Engineering

Raw features tell us *what happened*. Engineered features capture *what it means*.

| Engineered Feature | From | Captures |
|---|---|---|
| `dd_velocity_ratio` | dd_count_30d ÷ 90d avg | Is DD trending down? |
| `digital_engagement_score` | logins + recency | 0–100 engagement composite |
| `balance_stress_index` | volatility + overdrafts + NSFs | Financial friction |
| `relationship_depth_score` | tenure + products | How embedded? |
| `friction_score` | fee waivers + service calls | Dissatisfaction signal |

In [ ]:
train = cu.engineer_features(train_raw)
val   = cu.engineer_features(val_raw)
test  = cu.engineer_features(test_raw)

train[['dd_velocity_ratio','digital_engagement_score',
       'balance_stress_index','relationship_depth_score',
       'friction_score','churn_90d']].describe().round(3)

---
## 6. Model Training — AutoGluon

AutoGluon trains CatBoost, XGBoost, LightGBM, Neural Net, Random Forest — all at once.
Validates on held-out data and returns a **ranked leaderboard**. No hand-tuning.

> `time_limit=120` for live demo. Use 600–3600 for production quality.

In [ ]:
predictor, model_type = cu.train_model(train, val, time_limit=120)

In [ ]:
if model_type == 'autogluon':
    predictor.leaderboard(test[cu.FEATURE_COLS + [cu.TARGET_COL]], silent=True)

---
## 7. Evaluation

### 7a. ML Metrics — AUC-ROC, Precision-Recall, Calibration

> One metric is always misleading. Use all three, then translate to business impact.

In [ ]:
y_scores = cu.evaluate_model(predictor, test, model_type)

### 7b. Business Metrics — Lift, Precision@K, Cost-Benefit

In [ ]:
cu.plot_business_metrics(
    y_true               = test[cu.TARGET_COL].values,
    y_scores             = y_scores,
    avg_customer_value   = 1_200,
    retention_offer_cost = 75,
)

---
## 8. Global Explainability — SHAP Beeswarm

TreeSHAP decomposes each prediction into per-feature contributions.

**Reading the beeswarm:** position (right = increases churn), color (warm = high value), dot spread (variability across customers).

In [ ]:
shap_values, X_sample, shap_explainer = cu.compute_shap_values(
    predictor, test, model_type, sample_size=500
)

In [ ]:
cu.plot_shap_global(shap_values, X_sample, max_display=15)

In [ ]:
cu.plot_shap_bar(shap_values, X_sample, max_display=15)

---
## 9. DALEX — Model-Agnostic Explainability

**DALEX** (`dalex.drwhy.ai`) complements SHAP with a different set of lenses:

| Tool | What it shows |
|---|---|
| `model_parts()` | Permutation variable importance — works on *any* model |
| `model_profile()` | PDP / ALE — how each feature affects predictions across its full range |
| `predict_parts()` | Break-down plot — local attribution with interaction detection |
| Arena | Interactive multi-model comparison dashboard |

> **Rule of thumb:** Use SHAP for beeswarm + waterfall precision. Use DALEX for PDP/ALE, model comparison, and stakeholder dashboards.

In [ ]:
# Build DALEX explainer — wraps any predict_proba function model-agnostically
exp = cu.build_dalex_explainer(
    predictor,
    X_train    = train,
    y_train    = train[cu.TARGET_COL],
    model_type = model_type,
    label      = 'Churn Model',
)

In [ ]:
# Variable Importance — permutation-based, model-agnostic
# How much does AUC drop when each feature is randomly shuffled?
cu.plot_dalex_variable_importance(exp, n_features=15)

In [ ]:
# Partial Dependence Profiles — how does churn probability change across each feature's range?
cu.plot_dalex_pdp(
    exp,
    variables=['dd_count_30d', 'digital_engagement_score',
               'days_since_last_dd', 'friction_score']
)

In [ ]:
# ALE — preferred over PDP when features are correlated (avoids extrapolation)
cu.plot_dalex_ale(
    exp,
    variables=['dd_count_30d', 'digital_engagement_score',
               'balance_stress_index', 'friction_score']
)

---
## 10. Local Explainability — Mystery Customer Revealed

**Now we answer the opening question.** Two views: SHAP waterfall + DALEX break-down.

Same customer, same model — different lenses on *why* the prediction is what it is.

In [ ]:
cu.print_mystery_customer(mystery)

In [ ]:
# SHAP waterfall — exact TreeSHAP attribution
cu.plot_shap_local(
    shap_values, X_sample, shap_explainer, predictor,
    model_type=model_type, customer_idx=0
)

In [ ]:
# DALEX break-down — includes interaction effects between features
mystery_eng = cu.engineer_features(mystery.to_frame().T)
cu.plot_dalex_breakdown(exp, mystery_eng, label='Mystery Customer')

**Comparing SHAP waterfall vs DALEX break-down:**
- Both flag `days_since_last_dd` and `digital_engagement_score` as the top drivers
- DALEX break-down detects interaction effects — the *combination* of low DD + low logins compounds the risk
- SHAP is faster and exact for tree models; DALEX break-down is often more readable for non-technical stakeholders

---
## 11. ML + AI: The Narrative Layer

The model gives us precision. AI gives us communication. In production, SHAP/DALEX outputs feed an LLM prompt that generates a banker-readable action card.

In [ ]:
if model_type == 'autogluon':
    churn_prob = float(predictor.predict_proba(mystery_eng[cu.FEATURE_COLS])[1].values[0])
else:
    churn_prob = float(predictor.predict_proba(mystery_eng[cu.FEATURE_COLS])[:, 1][0])

cu.ai_narrative_explainer(mystery, churn_prob)

---
## Summary

| Step | Key Takeaway |
|------|--------------|
| **Data** | DD velocity + digital logins are the dominant banking churn signals |
| **Labeling** | Definition and look-forward window are as important as the algorithm |
| **EDA** | No primary DD = 3–4× higher churn rate |
| **Partitioning** | Always split by date — random splits leak future signal |
| **Feature Eng.** | Velocity ratios and composite scores beat raw counts |
| **Training** | AutoGluon ensemble outperforms any single model with zero tuning |
| **Evaluation** | ML metrics + business metrics — both required |
| **SHAP** | TreeSHAP is fast, exact, and produces beeswarm + waterfall |
| **DALEX** | Model-agnostic PDP/ALE + break-down + multi-model comparison |
| **ML + AI** | ML handles prediction at scale; AI handles explanation and action |

---
*Jack Henry Associates — Data Science Talk  |  github: churn_utils.py + churn_demo.ipynb*